Import de paquets

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import torch
from torch import randperm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, SubsetRandomSampler
from torchvision import datasets, transforms, models
from torchvision.transforms import v2
from torchmetrics.classification import BinaryAccuracy, BinaryConfusionMatrix
from torch.optim.lr_scheduler import StepLR
from sklearn.metrics import precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import my_helper
import logging
from tqdm import tqdm
import shutil
from glob import glob
import optuna
import optuna.trial
import mlflow
import mlflow.pytorch
from packaging import requirements
from pprint import pformat

In [ ]:
#logger = logging.getLogger("mlflow")
#logger.setLevel(logging.DEBUG)

seed pour la reproductibilité   

In [ ]:
def set_seed(seed=42):
    
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

vérification et choix du GPU si disponible

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du périphérique : {device}")

In [ ]:
# 1. Définition des chemins (ajustez-les si nécessaire)
source_dir = os.getcwd()+"\\data\\Cat_Dog_data"
target_dir = os.getcwd()+ "\\data\\Cat_Dog_ordered"

os.makedirs(target_dir, exist_ok=True)

# 2. Création de la nouvelle structure de dossier
for label in ["cat", "dog"]:
    os.makedirs(os.path.join(target_dir, label), exist_ok=True)

print("Début du déplacement des images...")

# 3. Recherche et déplacement des images
# Le pattern '*' englobe 'train' et 'test', et le deuxième '*' englobe les extensions (jpg, png, etc.)
search_pattern = os.path.join(source_dir, "*", "*", "*")
files = glob(search_pattern)

moved_count = 0

for file_path in files:
    # On vérifie qu'il s'agit bien d'un fichier et non d'un dossier
    if os.path.isfile(file_path):
        # On extrait les composants du chemin
        # Exemple : 'Cat_Dog_data/train/cat/img_1.jpg' -> ['Cat_Dog_data', 'train', 'cat', 'img_1.jpg']
        path_parts = file_path.split(os.sep)
        
        # Le label ('cat' ou 'dog') est l'avant-dernier élément
        label = path_parts[-2] 
        filename = path_parts[-1]
        
        if label in ["cat", "dog"]:
            # Pour éviter que des images de 'train' et 'test' ayant le même nom se détruisent,
            # on ajoute un préfixe (ex: 'train_img_1.jpg')
            subfolder = path_parts[-3] # 'train' ou 'test'
            new_filename = f"{subfolder}_{filename}"
            
            # Chemin de destination
            destination = os.path.join(target_dir, label, new_filename)
            
            # Déplacement du fichier
            shutil.move(file_path, destination)
            moved_count += 1

print(f"Terminé ! {moved_count} images ont été déplacées avec succès dans '{target_dir}'.")

In [ ]:
params = {
    "Experiment": "CNN_Cats&Dogs_Experiment",
    "data_dir": "data/Cat_Dog_ordered",
    "batch_size": 32,
    "num_workers": 6,
    "train_size": 0.8,
    "val_size": 0.1,
    "test_size": 0.1,
    "epochs": 20,
    "dropout": 0.5,
    "save_path": "best_model.pth"
}

Préparation des focntion de transformation et augmentation de données avec redimensionnement, 50% aléatoire de rotation de 15%, rotation horizontale de l'image, ainsi que normalisation de l'image pour train et redimensionnement et normalisation des données de test et validation.

In [ ]:
train_transforms = v2.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])
test_val_transforms = v2.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



In [ ]:

def get_data_loaders(data_dir, batch_size, train_size, valid_size, num_workers, train_transforms, test_val_transforms):
    train_dataset = datasets.ImageFolder(root=data_dir, transform=train_transforms)
    test_val_dataset = datasets.ImageFolder(root=data_dir, transform=test_val_transforms)
    
    n_tot = len(train_dataset)
    
    # 2. Calculer la taille des segments (80% / 10% / 10%)
    split_train = int(np.floor(train_size * n_tot))
    split_val = int(np.floor((train_size + valid_size) * n_tot))
    
    # 3. Mélanger et séparer les indices de manière aléatoire
    shuffled_indices = randperm(n_tot).tolist()
    
    train_idx = shuffled_indices[:split_train]
    valid_idx = shuffled_indices[split_train:split_val]
    test_idx = shuffled_indices[split_val:]
    
    # 4. Définir les samplers basés sur les indices
    train_sampler = SubsetRandomSampler(train_idx)
    valid_sampler = SubsetRandomSampler(valid_idx)
    test_sampler = SubsetRandomSampler(test_idx)
    
    # 5. Créer les DataLoaders
    # Note : On applique le train_dataset uniquement au train_loader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, num_workers=num_workers, persistent_workers=True, pin_memory=True,prefetch_factor=2)
    valid_loader = DataLoader(test_val_dataset, batch_size=batch_size, sampler=valid_sampler, num_workers=num_workers, persistent_workers=True, pin_memory=True,prefetch_factor=2)
    test_loader = DataLoader(test_val_dataset, batch_size=batch_size, sampler=test_sampler, num_workers=num_workers, persistent_workers=True, pin_memory=True,prefetch_factor=2)
    
    return train_loader, valid_loader, test_loader

# --- Instanciation des DataLoaders ---
train_dl, valid_dl, test_dl = get_data_loaders(
    data_dir=params["data_dir"],
    batch_size=params["batch_size"],
    train_size=params["train_size"],
    valid_size=params["val_size"],
    num_workers=params["num_workers"],
    train_transforms=train_transforms,
    test_val_transforms=test_val_transforms
)

# Regroupement dans le dictionnaire final
data_loaders = {
    'train': train_dl,
    'val': valid_dl,
    'test': test_dl
}

In [ ]:
classes = train_dl.dataset.classes
print(f"Classes : {classes}")
num_classes = len(classes)
print(f"Nombre de classes : {num_classes}")

Nombre de batchs pour train, val et test

In [ ]:
print(len(data_loaders['train']), len(data_loaders['val']), len(data_loaders['test']))

Test des loaders

In [ ]:
images, labels = next(iter(data_loaders["train"]))
fig, subs = plt.subplots(2, 10, figsize=(25, 4))
print(labels)
for i, sub in enumerate(subs.flatten()):
    my_helper.imshow(images[i], sub)
    sub.set_title([classes[labels[i]]])

In [ ]:
taille_batch = data_loaders["train"].batch_size

In [ ]:
images.size()

In [ ]:
batch = next(iter(data_loaders["train"]))

# Pour un tenseur unique ou une liste de tenseurs
if isinstance(batch, torch.Tensor):
    # element_size() donne la taille en octets d'un élément (ex: float32 = 4 octets)
    # nelement() donne le nombre total d'éléments dans le tenseur
    taille_octets = batch.element_size() * batch.nelement()
else:
    # Si le batch est une liste ou un dictionnaire (ex: (images, labels))
    taille_octets = sum(b.element_size() * b.nelement() for b in batch if isinstance(b, torch.Tensor))

# Conversion en KB
taille_kb = taille_octets / 1024
taille_mb = taille_kb / 1024

print(f"Taille du batch : {taille_mb:.2f} MB")

In [ ]:
image, label = next(iter(data_loaders["val"]))
my_helper.imshow(image[0],title=f"Label: {label[0]}")

In [ ]:
image, label = next(iter(data_loaders["test"]))
my_helper.imshow(image[0],title=f"Label: {label[0]}")

Conception du modèle

In [ ]:
class CNNScratch(nn.Module):
    def __init__(self):
        super(CNNScratch, self).__init__()
        
        self.features = nn.Sequential(
            # Bloc 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Bloc 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Bloc 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(512, 2) 
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
"""def train_model(model, criterion, optimizer, scheduler, num_epochs=params["num_workers"], save_path="best_model.pt"):
    best_val_loss = float('inf')
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        for inputs, labels in tqdm(data_loaders["train"], desc="Train", ncols=100):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        epoch_train_loss = running_loss / len(data_loaders["train"].dataset)
        epoch_train_acc = correct / total
        
        # Phase de Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in tqdm(data_loaders["val"], desc="Validation", ncols=100):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                
        epoch_val_loss = val_loss / len(data_loaders["val"].dataset)
        epoch_val_acc = val_correct / val_total
        
        if scheduler:
            scheduler.step()
            
        # Sauvegarde du meilleur modèle localement
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), save_path)
            
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        
        print(f"Époque {epoch+1}/{num_epochs} -> Train Loss: {epoch_train_loss:.4f} | Acc: {epoch_train_acc:.4f} || Val Loss: {epoch_val_loss:.4f} | Acc: {epoch_val_acc:.4f}")
        
    return history

def evaluate_final_metrics(model, path, loader):
    #Charge le meilleur modèle sauvegardé et calcule les métriques finales.
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    model.eval()
    
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    precision = precision_score(all_labels, all_preds, average='binary')
    recall = recall_score(all_labels, all_preds, average='binary')
    
    return acc, precision, recall, all_labels, all_preds"""


In [ ]:
"""criterion = nn.CrossEntropyLoss()
epochs = 10
if __name__ == "__main__":
    
# -- Expérience A : From Scratch --
    print("\n--- Lancement Expérience A : CNN From Scratch ---")
    scratch_model = CNNScratch().to(device)
    # Test de l'optimiseur Adam avec un Learning Rate Scheduler (Bonus)
    optimizer_scratch = optim.Adam(scratch_model.parameters(), lr=0.001)
    scheduler_scratch = optim.lr_scheduler.StepLR(optimizer_scratch, step_size=5, gamma=0.1)

history_scratch = train_model(scratch_model, criterion, optimizer_scratch, scheduler_scratch, epochs, "best_scratch.pth")"""

In [ ]:
"""plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history_scratch['train_loss'], label='Scratch Train Loss', linestyle='--')
plt.plot(history_scratch['val_loss'], label='Scratch Val Loss')
plt.title('Comparaison de la Loss')
plt.legend()"""

In [ ]:
"""plt.subplot(1, 2, 2)
plt.plot(history_scratch['train_acc'], label='Scratch Train Acc', linestyle='--')
plt.plot(history_scratch['val_acc'], label='Scratch Val Acc')
plt.title("Comparaison de l'Accuracy")
plt.legend()
plt.show()"""


In [ ]:
# print(evaluate_final_metrics(scratch_model, "best_scratch.pth", data_loaders["test"]))

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, acc_metric):
    model.train()
    acc_metric.reset()
    running_loss = 0.0
    for imgs, targets in tqdm(loader, desc="Train", ncols=100):
        imgs, targets = imgs.to(device,non_blocking=True), targets.to(device,non_blocking=True)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        preds = torch.argmax(logits, dim=1)
        acc_metric.update(preds, targets)
    return running_loss / len(loader.sampler), acc_metric.compute().item()

def evaluate(model, loader, criterion, acc_metric):
    model.eval()
    acc_metric.reset()
    running_loss = 0.0
    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc="Valid", ncols=100):
            imgs, targets = imgs.to(device), targets.to(device)
            logits = model(imgs)
            loss = criterion(logits, targets)
            running_loss += loss.item() * imgs.size(0)
            preds = torch.argmax(logits, dim=1)
            acc_metric.update(preds, targets)
    return running_loss / len(loader.sampler), acc_metric.compute().item()

def test_and_confusion(model, loader):
    model.eval()
    cm_metric = BinaryConfusionMatrix().to(device)
    test_acc.reset()
    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc="Test", ncols=100):
            imgs, targets = imgs.to(device), targets.to(device)
            logits = model(imgs)
            preds = torch.argmax(logits, dim=1)
            cm_metric.update(preds, targets)
            test_acc.update(preds, targets)
    cm = cm_metric.compute().cpu().numpy()
    acc = test_acc.compute().item()
    targets_cpu = targets.cpu().numpy()
    preds_cpu = preds.cpu().numpy()
    precision = precision_score(targets_cpu, preds_cpu, average='binary')
    recall = recall_score(targets_cpu, preds_cpu, average='binary')
    print(f"\nTest Accuracy : {acc:.2f}")
    return cm , acc, precision, recall

def plot_confusion_matrix(cm, classes):
    confusion_matrix = pd.DataFrame(cm, index=classes, columns=classes)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap="Blues", ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.savefig('confusion_matrix.png')
    plt.close()

def accuracy_by_class(cm, classes):
    print("\nAccuracy by class:\n")
    for i, class_name in enumerate(classes):
        correct_predictions = cm[i][i]
        total_predictions = cm[i].sum()
        accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
        print(f"    {class_name:11s}: {accuracy:.2f}")

def visualize_predictions(model, loader, classes, n_batches=1):
    model.eval()
    dataiter = iter(loader)
    for _ in range(n_batches):
        images, labels = next(dataiter)
        images, labels = images.to(device), labels.to(device)
        with torch.no_grad():
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
        fig, subs = plt.subplots(2, 10, figsize=(25, 4))
        for i, ax in enumerate(subs.flatten()):
            img = images[i].cpu().numpy()
            img = img / 2 + 0.5  # unnormalize
            ax.imshow(np.transpose(img, (1, 2, 0)))
            ax.set_title(f"{classes[preds[i]]} ({classes[labels[i]]})",
                         color=("green" if preds[i] == labels[i] else "red"))
            ax.axis("off")
        plt.savefig('prediction_visualized.png')
        plt.close()

In [ ]:
def suggest_hyperparameters(trial):
    # Obtain the learning rate on a logarithmic scale
    lr = trial.suggest_categorical("lr", [1e-4, 1e-3, 1e-2, 1e-1])

    # Obtain the optimizer to use by name
    optimizer_name = trial.suggest_categorical("optimizer_name", ["Adam", "SGD"])

    print(f"Suggested hyperparameters: \n{pformat(trial.params)}")
    return lr, optimizer_name

In [ ]:
def objective(trial):
    print("\n********************************\n")
    best_val = float("inf")
    with mlflow.start_run(nested=True):
        mlflow.log_param("device", device)
        mlflow.log_params(params)
        lr, optimizer_name = suggest_hyperparameters(trial)
        mlflow.log_params(trial.params)
        print(f"Training on {device}")
        criterion = nn.CrossEntropyLoss()
        train_acc = BinaryAccuracy().to(device)
        valid_acc = BinaryAccuracy().to(device)
        global test_acc
        test_acc = BinaryAccuracy().to(device)
        modele = CNNScratch().to(device)
        if optimizer_name == "Adam":
            optimizer = optim.Adam(modele.parameters(), lr=lr)
        if optimizer_name == "SGD":
            optimizer = optim.SGD(modele.parameters(), lr=lr,momentum=0.9)
        scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
        for epoch in range(1, params["epochs"] + 1):
            tr_loss, tr_acc = train_one_epoch(modele, data_loaders["train"], criterion, optimizer, train_acc)
            val_loss, val_acc = evaluate(modele, data_loaders["val"], criterion, valid_acc)
            print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")
            mlflow.log_metrics({"train_loss": tr_loss, "train_accuracy": tr_acc, "val_loss": val_loss, "val_accuracy": val_acc}, step=epoch)
            if val_loss < best_val:
                best_val = val_loss
                torch.save(modele.state_dict(), params["save_path"])
                print("New best model saved!")
            scheduler.step()
    
        mlflow.pytorch.log_model(modele, name="final_model", pip_requirements="requirements.txt")
    
        # Final test + confusion matrix
        print("\nEvaluating on test set …")
        cm, test_acc, precision, recall = test_and_confusion(modele, data_loaders["test"])
        mlflow.log_metric("test_accuracy", test_acc)
        mlflow.log_metric("test_precision", precision)
        mlflow.log_metric("test_recall", recall)
        plot_confusion_matrix(cm, classes)
        mlflow.log_artifact(os.getcwd()+"\\confusion_matrix.png", artifact_path="confusion_matrix")
        accuracy_by_class(cm,classes)

        visualize_predictions(modele,data_loaders['test'],classes)
        mlflow.log_artifact(os.getcwd()+"\\prediction_visualized.png", artifact_path="prediction_visualized")
        return test_acc

In [ ]:
def main():
    mlflow.set_tracking_uri("http://localhost:5000")
    mlflow.set_experiment(params["Experiment"])
    mlflow.config.enable_system_metrics_logging()
    print("✓ Successfully connected to MLflow!")
    with open('requirements.txt', 'r') as file:
        requirements_info = [line.strip() for line in file]
    
    study = optuna.create_study(study_name="cifar10-torchmetrics-mlfow", direction="maximize")
    study.optimize(objective, n_trials=10)

    # Print optuna study statistics
    print("\n++++++++++++++++++++++++++++++++++\n")
    print("Study statistics: ")
    print("  Number of finished trials: ", len(study.trials))

    print("Best trial:")
    trial = study.best_trial

    print("  Trial number: ", trial.number)
    print("  Loss (trial value): ", trial.value)

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))

        

In [32]:
main()


[I 2026-05-23 17:27:14,328] A new study created in memory with name: cifar10-torchmetrics-mlfow
2026/05/23 17:27:14 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


✓ Successfully connected to MLflow!

********************************

Suggested hyperparameters: 
{'lr': 0.026171454806653588, 'optimizer_name': 'SGD'}
Training on cuda


Train:  19%|██████████                                            | 117/625 [00:10<00:43, 11.66it/s]
2026/05/23 17:27:25 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/05/23 17:27:25 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
[W 2026-05-23 17:27:25,125] Trial 0 failed with parameters: {'lr': 0.026171454806653588, 'optimizer_name': 'SGD'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\dodzi\Documents\Codes Python\cnn-catsdogs-AHNERTDodzi\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\dodzi\AppData\Local\Temp\ipykernel_25740\139999909.py", line 22, in objective
    tr_loss, tr_acc = train_one_epoch(modele, data_loaders["train"], criterion, optimizer, train_acc)
                      ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

🏃 View run nebulous-skink-990 at: http://localhost:5000/#/experiments/4/runs/070ec1b68b3a4de38353d6f1654dc0c8
🧪 View experiment at: http://localhost:5000/#/experiments/4


KeyboardInterrupt: 

In [ ]:
print(study.)

NameError: name 'trial' is not defined